In [3]:
import json
import pandas as pd
import yaml

json_file_name = input("Введите название json-файла (например: 2026-03-19_17-13-07): ")
with open(json_file_name+".json", "r") as JSON:
    data_dict = json.load(JSON)

df = pd.DataFrame(data_dict["hourly"])
with open("variant_4.yml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)
    df["city_id"] = config["entity"]["city_id"]

#  ПРОВЕРЯЕМ ИСХОДНЫЕ ДАННЫЕ


# Проверка на некорректные данные в столбцах
print("Пропуски по столбцам: ")
display(df.isna().sum())
print("\nКоличество полных дубликатов строк:", df.duplicated().sum())
print("Количество дубликатов по time:", df.duplicated(subset=["time"]).sum(), '\n')

# Приведение времени к datetime, влажности к float
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df["relative_humidity_2m"] = pd.to_numeric(df["relative_humidity_2m"], errors="coerce", downcast="float")
display(df)
print("Пропуски после приведения типов:")
display(df.isna().sum())


#   ОЧИСТКА


clean_df = df.copy()

# 1) Удаление строк без времени
clean_df = clean_df.dropna(subset=["time"])

# 2) Удаление дубликатов по времени, оставляя первую запись
clean_df = clean_df.drop_duplicates(subset=["time"], keep="first")

# 3) Работа с пропусками
clean_df["precipitation"] = clean_df["precipitation"].fillna(0)
clean_df = clean_df.dropna(subset=["temperature_2m"])

# 4) Сортировка по времени
clean_df = clean_df.sort_values("time").reset_index(drop=True)

# 5) Переименование столбцов
clean_df.columns = ["Время, ГГГГ-ММ-ДД ЧЧ:ММ:СС", 
              "Температура, °C", 
              "Влажность, %",
              "Осадки, мм",
              "Скорость ветра, км/ч",
              "ID города"
            ]


# ЧТО ОСТАЛОСЬ ПОСЛЕ ОЧИСТКИ


print("head():")
display(clean_df.head())

print("\nshape:", clean_df.shape)
print("\ndtypes:")
print(clean_df.dtypes)

print("\nПропуски:")
print(clean_df.isna().sum())

print("\nДубликаты по time:", clean_df.duplicated(subset=["Время, ГГГГ-ММ-ДД ЧЧ:ММ:СС"]).sum())
print("Минимальная дата:", clean_df["Время, ГГГГ-ММ-ДД ЧЧ:ММ:СС"].min())
print("Максимальная дата:", clean_df["Время, ГГГГ-ММ-ДД ЧЧ:ММ:СС"].max())
print("Таблица пустая?", clean_df.empty)


clean_df.to_csv(json_file_name+".csv", sep=",", index=False, encoding="utf-8")

Введите название json-файла (например: 2026-03-19_17-13-07):  2026-03-19_17-13-07


Пропуски по столбцам: 


time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
city_id                 0
dtype: int64


Количество полных дубликатов строк: 0
Количество дубликатов по time: 0 



,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,city_id
0,2026-03-01 00:00:00,7.2,82.0,0.0,11.9,GB_LON
1,2026-03-01 01:00:00,6.9,83.0,0.0,11.6,GB_LON
2,2026-03-01 02:00:00,6.9,85.0,0.0,11.8,GB_LON
3,2026-03-01 03:00:00,7.2,86.0,0.0,12.2,GB_LON
4,2026-03-01 04:00:00,7.4,87.0,0.0,13.2,GB_LON
...,...,...,...,...,...,...
451,2026-03-19 19:00:00,10.6,69.0,0.0,7.6,GB_LON
452,2026-03-19 20:00:00,9.6,72.0,0.0,6.7,GB_LON
453,2026-03-19 21:00:00,8.7,75.0,0.0,5.9,GB_LON
454,2026-03-19 22:00:00,8.1,77.0,0.0,5.4,GB_LON


Пропуски после приведения типов:


time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
city_id                 0
dtype: int64

head():


,"Время, ГГГГ-ММ-ДД ЧЧ:ММ:СС","Температура, °C","Влажность, %","Осадки, мм","Скорость ветра, км/ч",ID города
0,2026-03-01 00:00:00,7.2,82.0,0.0,11.9,GB_LON
1,2026-03-01 01:00:00,6.9,83.0,0.0,11.6,GB_LON
2,2026-03-01 02:00:00,6.9,85.0,0.0,11.8,GB_LON
3,2026-03-01 03:00:00,7.2,86.0,0.0,12.2,GB_LON
4,2026-03-01 04:00:00,7.4,87.0,0.0,13.2,GB_LON



shape: (456, 6)

dtypes:
Время, ГГГГ-ММ-ДД ЧЧ:ММ:СС    datetime64[ns]
Температура, °C                      float64
Влажность, %                         float32
Осадки, мм                           float64
Скорость ветра, км/ч                 float64
ID города                             object
dtype: object

Пропуски:
Время, ГГГГ-ММ-ДД ЧЧ:ММ:СС    0
Температура, °C               0
Влажность, %                  0
Осадки, мм                    0
Скорость ветра, км/ч          0
ID города                     0
dtype: int64

Дубликаты по time: 0
Минимальная дата: 2026-03-01 00:00:00
Максимальная дата: 2026-03-19 23:00:00
Таблица пустая? False
